
%md
# 02 | Silver Layer — Vehicle Sales Transformation

**Project:** US Vehicle Sales Analytics Pipeline  
**Notebook:** `02_silver_transformation`  
**Source:** `workspace.bronze.raw_vehicle_sales`  
**Target:** `workspace.silver.vehicle_monthly_sales`  
**Technology:** PySpark | Delta Lake | Delta MERGE

---

## Objective

Transform raw passenger-car sales snapshots into a clean, normalized monthly sales dataset.

## Transformations

1. Read the latest ingested Bronze snapshot.
2. Convert monthly columns into rows using unpivoting.
3. Standardize vehicle model names.
4. Convert sales quantities into numeric values.
5. Validate model names, dates and nonnegative sales quantities.
6. Exclude months without reported sales values.
7. Deduplicate records using model and sales month.
8. Apply Delta MERGE for inserts, updates and unchanged records.

## Business Key

`model + sales_month`

This key assumes each source row represents one unique vehicle model. We will validate that assumption.

## Incremental Processing

The source provides periodic file snapshots, not CDC events.

The Silver layer compares the latest available source snapshot against existing records and uses Delta MERGE to apply changes.

## Data Quality Rules

- Vehicle model must not be NULL or empty.
- Sales month must be valid.
- Units sold must be a nonnegative integer.
- Duplicate model-month records must be detected.
- Missing monthly sales must not be treated as zero.
- Invalid data must fail validation rather than silently enter Silver.

## Expected Output

**Target Table:** `workspace.silver.vehicle_monthly_sales`

One valid row per vehicle model and reporting month.

The exact row count depends on the number of nonblank monthly sales values.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_table = "vehicle_sales.bronze.raw_vehicle_sales"
silver_table = "vehicle_sales.silver.vehicle_monthly_sales"


#get the latest successfully ingested batch 

latest_batch = (spark.table(bronze_table)
                .select("batch_id", "ingest_ts")
                .distinct()
                .orderBy(
                    F.desc("ingest_ts"),
                    F.desc("batch_id")
                )
                .first()
                )

if latest_batch is None:
    raise Exception("No data in bronze table")

batch_id = latest_batch["batch_id"]

bronze_df = (
    spark.table(bronze_table) 
    .filter(F.col("batch_id") == batch_id)
)


print("processing batch", batch_id)   
print("bronze records:", bronze_df.count())
display(bronze_df.limit(10))





In [0]:
months = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

required_columns = {"Model", *months}

missing_columns = required_columns - set(bronze_df.columns)

if missing_columns:
    raise Exception(f"Missing required columns: {missing_columns}")


# convert 12 monthly column into model-month rows 

monthly_values = F.array(*[
    F.struct(
        F.lit(month_num).alias("month_num"),
        F.col(f"{month}").cast("string").alias("raw_sales")
    )
    for month_num, month in enumerate(months, 1)
])

unpivot_df = (
    bronze_df 
    .select(
        F.trim(F.col("Model")).alias("model"),
        "batch_id",
        "ingest_ts",
        F.explode(monthly_values).alias("month_data")
    )
    .select(
        "model",
        F.to_date(
            F.format_string(
                "2026-%02d-01",
                F.col("month_data.month_num")
            )
            ).alias("sales_month"), 
        F.trim(
            F.col("month_data.raw_sales")
        ).alias("raw_sales"),
        "batch_id",
        "ingest_ts"
        )
    )

display(unpivot_df.limit(20))

In [0]:
clean_df = (
    unpivot_df
    .withColumn(
        "sales_text",
        F.regexp_replace("raw_sales", ",", "")
    )
    .withColumn(
        "is_missing",
        F.col("sales_text").isNull()
        | (F.col("sales_text") == "")
    )
    .withColumn(
        "is_valid_number",
        F.col("sales_text").rlike(r"^[0-9]+$")
    )
)

# Reject malformed values and missing model names
invalid_df = clean_df.filter(
    F.col("model").isNull()
    | (F.col("model") == "")
    | (
        ~F.col("is_missing")
        & ~F.col("is_valid_number")
    )
)

invalid_count = invalid_df.count()

if invalid_count > 0:
    display(invalid_df)
    raise ValueError(
        f"Data quality failed: {invalid_count} invalid rows"
    )

# Exclude months without reported values
silver_df = (
    clean_df
    .filter(~F.col("is_missing"))
    .withColumn(
        "units_sold",
        F.expr("try_cast(sales_text AS BIGINT)")
    )
    .select(
        "model",
        "sales_month",
        "units_sold",
        "batch_id",
        "ingest_ts"
    )
)

# Check for numeric overflow
if silver_df.filter(F.col("units_sold").isNull()).limit(1).count():
    raise ValueError("Sales value exceeds BIGINT range")

display(silver_df.limit(20))

In [0]:
duplicate_df = (
    silver_df
    .groupBy("model", "sales_month")
    .count()
    .filter("count > 1")
)

if duplicate_df.limit(1).count() > 0:
    display(duplicate_df)
    raise ValueError("Duplicate model-month keys detected")

silver_df = silver_df.dropDuplicates(
    ["model", "sales_month"]
)

# Create the target if it doesn't exist
if not spark.catalog.tableExists(silver_table):

    (
        silver_df.write
        .format("delta")
        .mode("error")
        .saveAsTable(silver_table)
    )

    print("Initial Silver table created")

else:

    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("t")
        .merge(
            silver_df.alias("s"),
            """
            t.model = s.model
            AND t.sales_month = s.sales_month
            """
        )
        .whenMatchedUpdate(
            condition=(
                "NOT (t.units_sold <=> s.units_sold)"
            ),
            set={
                "units_sold": "s.units_sold",
                "batch_id": "s.batch_id",
                "ingest_ts": "s.ingest_ts"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver MERGE completed")

In [0]:
%sql
SELECT
    sales_month,
    COUNT(*) AS model_count,
    SUM(units_sold) AS total_units_sold
FROM vehicle_sales.silver.vehicle_monthly_sales
GROUP BY sales_month
ORDER BY sales_month;

In [0]:
%sql
SELECT
    model,
    sales_month,
    COUNT(*) AS records
FROM vehicle_sales.silver.vehicle_monthly_sales
GROUP BY model, sales_month
HAVING COUNT(*) > 1;